In [1]:
import pandas as pd
import torch
import numpy as np
import torch.nn.functional as F
import re
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from datasets import Dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer, 
    set_seed
)
import gc

# 1. 초기화 및 메모리 정리
gc.collect()
torch.cuda.empty_cache()

# 2. 하이퍼파라미터 세팅 (요청하신 기준 완벽 반영)
MODEL_NAME = "klue/roberta-large"
MAX_LENGTH = 128
TARGET_BATCH_SIZE = 32
PER_DEVICE_BATCH = 1  # 메모리 확보를 위해 분할 (8 * 4 = 32)
ACCUMULATION_STEPS = TARGET_BATCH_SIZE // PER_DEVICE_BATCH 

LEARNING_RATE = 1e-5
EPOCHS = 5
WEIGHT_DECAY = 0.01
ADAM_EPSILON = 1e-5
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1  # 전체 Step의 10%를 Linear Warmup으로 설정
SEED = 42

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 3. 데이터셋 로드 및 준비
print("데이터셋을 로드합니다...")
final_df = pd.read_csv('final_train_data.csv')

# 데이터를 무작위로 섞어주기
final_df = final_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

# Train / Valid 분할
train_df, val_df = train_test_split(final_df, test_size=0.2, random_state=SEED, stratify=final_df['label'])

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# 토크나이저 로드 (뒷부분 핵심 문맥 보존을 위해 left 자르기 설정)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.truncation_side = 'left'

def tokenize_function(examples):
    return tokenizer(
        examples["conversation"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)

# 불필요한 컬럼 제거 및 텐서 변환
cols_to_remove = ["conversation"]
if "__index_level_0__" in tokenized_train.column_names:
    cols_to_remove.append("__index_level_0__")

tokenized_train = tokenized_train.remove_columns(cols_to_remove).rename_column("label", "labels").with_format("torch")
tokenized_val = tokenized_val.remove_columns(cols_to_remove).rename_column("label", "labels").with_format("torch")

# 4. 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=5
).to(device)

model.config.use_cache = False  # 체크포인팅 에러 방지

# 5. TrainingArguments (모든 조건 적용)
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    
    # Batch Size 세팅 (메모리 방어를 위해 1로 설정, 누적으로 32 맞춤)
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=32,
    
    # 💡 [핵심 1] 메모리 절반 압축 (RoBERTa-Large에서는 필수 권장)
    fp16=True,  
    
    # 💡 [핵심 2] 옵티마이저 설정 (아래 둘 중 하나 선택)
    
    # 옵션 A (추천): 8-bit AdamW (bitsandbytes 설치 필요)
    # optim="adamw_bnb_8bit", 
    
    # 옵션 B: 순정 AdamW (클라우드 GPU 메모리가 24GB 이상일 때)
    optim="adamw_torch", 
    
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    
    eval_strategy="epoch",  
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=10,
    seed=SEED,
    load_best_model_at_end=True,  
    metric_for_best_model="f1_macro",
)

# 6. Loss, Accuracy, F1 계산 함수
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    preds = np.argmax(predictions, axis=1)
    
    accuracy = accuracy_score(labels, preds)
    f1_macro = f1_score(labels, preds, average='macro')
    f1_weighted = f1_score(labels, preds, average='weighted')
    
    # Trainer가 Loss는 기본으로 출력하므로, Acc와 F1을 추가로 반환합니다.
    return {
        "accuracy": accuracy,
        "f1_macro": f1_macro,
        "f1_weighted": f1_weighted
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

# 7. 학습 시작
print("🚀 학습을 시작합니다...")
trainer.train()
print("✅ 학습 완료!")

# 8. 테스트 문장 및 예측 함수
id2label = {0: '협박 대화', 1: '갈취 대화', 2: '직장 내 괴롭힘 대화', 3: '기타 괴롭힘 대화', 4: '일반 대화'}

def standard_preprocess(text):
    if not isinstance(text, str): return ""
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'http\S+|www\S+|mailto:\S+', '', text)
    text = re.sub(r'[^가-힣0-9a-zA-Zㄱ-ㅎㅏ-ㅣ\s.?!,]', '', text)
    return text.strip()

def predict_sentence_with_confidence(sentence):
    clean_sentence = standard_preprocess(sentence)
    inputs = tokenizer(
        clean_sentence, 
        return_tensors="pt", 
        max_length=MAX_LENGTH, 
        padding="max_length", 
        truncation=True
    ).to(device)
    
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=-1)
        max_prob, predicted_class_id = torch.max(probs, dim=-1)
        
    return id2label[predicted_class_id.item()], max_prob.item() * 100

# 9. 최종 결과 출력
print("\n" + "="*50)
print("📝 확신도가 포함된 테스트 문장 예측 결과")
print("="*50)

additional_test_sentences = [
    "너 어제 내가 말한 거 왜 처리 안 했어? 죄송해요 진짜 어제 몸이 너무 안 좋아서 그랬어요. 내 인내심 테스트하지 마. 오늘 밤까지 안 보내면 너네 집 주소 아는 사람 보낼 거야. 진짜 죽기 싫으면 제대로 해라.",
    "야 이번에 부모님한테 용돈 받았다며? 응 근데 이건 진짜 학원비 내야 하는 돈이라서 안 돼. 야 학원비는 나중에 내고 일단 좀 내놔봐. 좋은 말로 할 때 주는 게 서로 편하지 않겠냐? 내일까지 이자 붙여서 가져와.",
    "이대리 이 기획안 제정신으로 쓴 거야? 초등학생도 이것보다는 잘 쓰겠다. 죄송합니다 다시 수정해서 보고드리겠습니다. 다시 한다고 뭐가 달라져? 너 같은 무능한 애가 우리 팀에 있다는 게 진짜 민폐다 그냥 오늘 당장 사표 쓰고 나가.",
    "야 쟤 또 저 옷 입고 왔네? 진짜 냄새나게 생겼다. 야 들리겠어 그만해. 들으라고 하는 소리야 너는 거울도 안 보고 사냐? 우리 쪽으로 오지 마 재수 없으니까.",
    "팀장님 오늘 회의록 정리해서 메일로 보내드렸습니다. 아 방금 확인했습니다 고생 많으셨어요. 별말씀을요 추가로 수정할 부분 있으시면 퇴근 전에 말씀해 주세요!"
]

for text in additional_test_sentences:
    pred_label, confidence = predict_sentence_with_confidence(text)
    print(f"문장: {text[:40]}...")
    print(f"결과: {pred_label} (확신도: {confidence:.2f}%)\n")

데이터셋을 로드합니다...


config.json:   0%|          | 0.00/547 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/375 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ 학습 완료!

📝 확신도가 포함된 테스트 문장 예측 결과
문장: 너 어제 내가 말한 거 왜 처리 안 했어? 죄송해요 진짜 어제 몸이 너무...
결과: 협박 대화 (확신도: 99.90%)

문장: 야 이번에 부모님한테 용돈 받았다며? 응 근데 이건 진짜 학원비 내야 하...
결과: 갈취 대화 (확신도: 99.44%)

문장: 이대리 이 기획안 제정신으로 쓴 거야? 초등학생도 이것보다는 잘 쓰겠다....
결과: 직장 내 괴롭힘 대화 (확신도: 99.91%)

문장: 야 쟤 또 저 옷 입고 왔네? 진짜 냄새나게 생겼다. 야 들리겠어 그만해...
결과: 기타 괴롭힘 대화 (확신도: 99.94%)

문장: 팀장님 오늘 회의록 정리해서 메일로 보내드렸습니다. 아 방금 확인했습니다...
결과: 일반 대화 (확신도: 87.76%)



In [4]:
import numpy as np
import pandas as pd
from scipy.special import softmax
from sklearn.metrics import classification_report

print("🚀 검증 데이터 기반 최종 상세 평가 및 확신도 분석 중...")

# 1. 평가 데이터에 대해 예측 수행 (Logit 값 추출)
predictions = trainer.predict(tokenized_val)
logits = predictions.predictions
true_labels = predictions.label_ids

# 2. Logit을 Softmax로 변환하여 확률(확신도) 계산
probs = softmax(logits, axis=1)
pred_labels = np.argmax(probs, axis=1)         # 가장 높은 확률의 라벨(0~4)
confidences = np.max(probs, axis=1) * 100      # 가장 높은 확률의 퍼센트(%)

# 3. 기본 성적표 출력
target_names = ['협박(0)', '갈취(1)', '직장내괴롭힘(2)', '기타괴롭힘(3)', '일반대화(4)']
print("\n" + "="*50)
print("📊 [클래스별 상세 성적표]")
print("="*50)
print(classification_report(true_labels, pred_labels, target_names=target_names))

# =====================================================================
# 🌟 [추가된 기능] 모델의 확신도를 바탕으로 한 '오답 노트' 생성
# =====================================================================

# 원본 텍스트를 가져와서 데이터프레임으로 묶기 (val_df 사용)
error_analysis_df = pd.DataFrame({
    '문장': val_df['conversation'].values,
    '정답': [target_names[i] for i in true_labels],
    '예측': [target_names[i] for i in pred_labels],
    '확신도(%)': np.round(confidences, 2)
})

# 4-1. 모델이 틀린 문제만 필터링
wrong_preds = error_analysis_df[error_analysis_df['정답'] != error_analysis_df['예측']]

# 4-2. 확신도가 높은 순서대로 정렬 (완전 확신했는데 틀린 '최악의 오답')
worst_mistakes = wrong_preds.sort_values(by='확신도(%)', ascending=False)

print("\n" + "="*80)
print("🚨 [핵심 오답 노트] 모델이 '완벽하게 확신했는데' 틀린 Top 5 문장")
print("   (이 문장들을 보면 모델이 어떤 패턴을 오해하고 있는지 알 수 있습니다!)")
print("="*80)

# 문장이 길어도 잘리지 않고 다 보이게 설정
pd.set_option('display.max_colwidth', None)
print(worst_mistakes.head(5).to_string(index=False))
pd.reset_option('display.max_colwidth')

# 전체 오답 노트를 CSV 파일로 저장
worst_mistakes.to_csv("model_worst_mistakes.csv", index=False, encoding='utf-8-sig')
print("\n✅ 오답 노트가 'model_worst_mistakes.csv'로 저장되었습니다.")

🚀 검증 데이터 기반 최종 상세 평가 및 확신도 분석 중...



📊 [클래스별 상세 성적표]
              precision    recall  f1-score   support

       협박(0)       0.91      0.91      0.91       178
       갈취(1)       0.90      0.89      0.89       195
   직장내괴롭힘(2)       0.98      0.96      0.97       194
    기타괴롭힘(3)       0.90      0.93      0.91       202
     일반대화(4)       1.00      1.00      1.00       179

    accuracy                           0.94       948
   macro avg       0.94      0.94      0.94       948
weighted avg       0.94      0.94      0.94       948


🚨 [핵심 오답 노트] 모델이 '완벽하게 확신했는데' 틀린 Top 5 문장
   (이 문장들을 보면 모델이 어떤 패턴을 오해하고 있는지 알 수 있습니다!)
                                                                                                                                                                                                                                                                                                                                                                            문장        정답       예측    확신도(%)
          

In [3]:
import pandas as pd
import torch
from tqdm import tqdm

# 1. 데이터 로드 (파일 경로를 확인해 주세요)
# 사용자님이 말씀하신 idx, conversation 컬럼이 있는 파일입니다.
test_df = pd.read_csv('./data/test.csv') 

# 2. 추론 모드 설정
model.eval()
submission_list = []

print("🚀 Kaggle 제출용 추론 시작 (컬럼명: idx, target)...")

for _, row in tqdm(test_df.iterrows(), total=len(test_df)):
    # test.csv의 컬럼명 사용
    idx = row['idx']
    text = row['conversation']
    
    # 학습 시와 동일한 전처리 적용 (필수)
    if 'standard_preprocess' in globals():
        text = standard_preprocess(text)
    
    inputs = tokenizer(
        text,
        return_tensors="pt",
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True
    ).to(device)

    with torch.no_grad():
        outputs = model(**inputs)
        # 숫자로 된 클래스 번호(0, 1, 2, 3, 4)를 바로 가져옵니다.
        predicted_class_id = torch.argmax(outputs.logits, dim=-1).item()

    # 사이트 안내에 맞게 idx와 target 컬럼 구성
    submission_list.append({
        'idx': idx,
        'target': predicted_class_id
    })

# 3. submission.csv 저장
submission_df = pd.DataFrame(submission_list)
submission_df.to_csv('submission.csv', index=False)

print("\n✅ submission.csv 생성 완료!")
print(submission_df.head()) # 상위 5개 행을 출력해 형식을 확인합니다.

🚀 Kaggle 제출용 추론 시작 (컬럼명: idx, target)...


100%|██████████| 500/500 [00:13<00:00, 37.69it/s]


✅ submission.csv 생성 완료!
     idx  target
0  t_000       1
1  t_001       2
2  t_002       2
3  t_003       3
4  t_004       3
